# Fabric Workspace Inventory – Core + Admin REST API

## Purpose
This notebook creates a complete inventory of all items within a Microsoft Fabric workspace, including Lakehouses, Warehouses, Notebooks, Pipelines, Semantic Models, Reports, Dataflows, and other supported assets.

The inventory captures essential metadata such as:
- Item Name
- Item Type
- Item ID
- Description
- Owner (Created By)*
- Last Modified Date*
- Workspace Information

This inventory can be used to:
- Track workspace assets
- Identify unused or obsolete objects
- Support governance and documentation
- Monitor ownership and modification history
- Periodically capture workspace snapshots for auditing

> **Note:** *Owner (Created By) and Last Modified information are available only when the notebook is executed by a Fabric Administrator.*

---

## APIs Used

This notebook combines information from two Microsoft Fabric REST APIs.

### 1. Core API – List Workspace Items
Retrieves the complete list of items available within the specified workspace.

```
GET /v1/workspaces/{workspaceId}/items
```

Provides:
- Item Name
- Item Type
- Item ID
- Description

---

### 2. Admin API – List Items
Retrieves administrative metadata for workspace items.

```
GET /v1/admin/items?workspaceId={workspaceId}
```

Provides:
- Created By
- Last Modified Date
- Additional administrative metadata

The Admin API is queried only for the specified workspace and is **not** used for tenant-wide discovery.

---

## Required Permissions

| Capability | Minimum Requirement |
|------------|---------------------|
| View workspace inventory | Viewer role on the workspace |
| Retrieve **Created By** and **Last Modified** metadata | Fabric Administrator |
| Save inventory snapshot to a Delta table | Contributor (or higher) with a default Lakehouse attached |

---

## Graceful Degradation

If the notebook is executed by a user who is **not** a Fabric Administrator:

- The Core API inventory is still generated successfully.
- Administrative metadata (Created By and Last Modified) is skipped automatically.
- The notebook continues execution without failure.

---

## Output

The final inventory can optionally be written to a Delta table, enabling:

- Historical workspace snapshots
- Change tracking over time
- Asset governance
- Cleanup and lifecycle management
- Reporting and auditing


## 1. Parameters

In [ ]:
# PARAMETERS
workspace_id = ""              # Leave blank to use the workspace this notebook runs in
save_to_lakehouse = True       # Set True to append this run's snapshot to a Delta table
lakehouse_table_name =         "workspace_inventory_snapshot"
stale_cutoff_days = 90         # Flag items not modified in more than this many days

## 2. Setup

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import notebookutils

BASE_URL = "https://api.fabric.microsoft.com/v1"

In [ ]:
# Resolve target workspace and acquire a delegated auth token.
# getToken("pbi") issues a token scoped to the signed-in user's own permissions -
# if that user is a Fabric Administrator, the same token also works against the /admin/ endpoints.
if not workspace_id:
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

print(f"Target workspace: {workspace_id}")

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

## 3. API helper with pagination + throttling retry

In [ ]:
def call_fabric_api(url, headers, max_retries=5):
    """
    GET wrapper for Fabric REST API calls.
    Retries on 429 (throttling) using the Retry-After header, and raises a clear
    PermissionError on 401/403 so calling code can degrade gracefully.
    """
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Throttled by API. Waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            raise PermissionError(f"Access denied ({resp.status_code}) calling {url}: {resp.text}")
        else:
            resp.raise_for_status()
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

## 4. Layer 1 - Core Items API (works with Viewer role)

In [ ]:
def get_all_items(workspace_id):
    """Fetch all items in a workspace via the Core Items API."""
    items, url = [], f"{BASE_URL}/workspaces/{workspace_id}/items"
    while url:
        data = call_fabric_api(url, headers)
        items.extend(data.get("value", []))
        cont_token = data.get("continuationToken")
        url = f"{BASE_URL}/workspaces/{workspace_id}/items?continuationToken={cont_token}" if cont_token else None
    return items

core_items = get_all_items(workspace_id)
print(f"Core Items API returned {len(core_items)} items.")

df_core = pd.DataFrame(core_items)
if not df_core.empty:
    df_core = df_core[["id", "displayName", "type", "description"]].rename(columns={"displayName": "name"})
else:
    df_core = pd.DataFrame(columns=["id", "name", "type", "description"])

## 5. Layer 2 - Admin Items API (Created By + Last Modified, requires Fabric Admin)

In [ ]:
admin_available = True

def extract_creator(item):
    principal = item.get("creatorPrincipal") or {}
    return principal.get("userDetails", {}).get("userPrincipalName") or principal.get("displayName")

try:
    def get_admin_items(workspace_id):
        """Fetch owner/last-modified metadata via the Admin Items API (Fabric Administrator only)."""
        items, url = [], f"{BASE_URL}/admin/items?workspaceId={workspace_id}"
        while url:
            data = call_fabric_api(url, headers)
            items.extend(data.get("itemEntities", []))
            cont_token = data.get("continuationToken")
            url = f"{BASE_URL}/admin/items?workspaceId={workspace_id}&continuationToken={cont_token}" if cont_token else None
        return items

    admin_items = get_admin_items(workspace_id)
    print(f"Admin Items API returned {len(admin_items)} items.")

    df_admin = pd.DataFrame([{
        "id": i["id"],
        "created_by": extract_creator(i),
        "last_modified": i.get("lastUpdatedDate"),
        "state": i.get("state"),
        "capacity_id": i.get("capacityId"),
    } for i in admin_items])

except PermissionError as e:
    admin_available = False
    print("Admin API not accessible - you are likely not a Fabric Administrator.")
    print(f"Details: {e}")
    print("Continuing with Core Items data only (no Created By / Last Modified columns).")
    df_admin = pd.DataFrame(columns=["id", "created_by", "last_modified", "state", "capacity_id"])

## 6. Combine, summarize, and flag stale objects

In [ ]:
df_final = df_core.merge(df_admin, on="id", how="left")
df_final["snapshot_time_utc"] = datetime.now(timezone.utc).isoformat()
df_final = df_final.sort_values(["type", "name"]).reset_index(drop=True)

print(f"Total objects in workspace: {len(df_final)}")
display(df_final)

In [ ]:
summary = (
    df_final.groupby("type")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print("Object counts by type:")
display(summary)

In [ ]:
if admin_available and not df_final["last_modified"].isna().all():
    df_final["last_modified_dt"] = pd.to_datetime(df_final["last_modified"], errors="coerce", utc=True)
    df_final["days_since_modified"] = (pd.Timestamp.now(tz="UTC") - df_final["last_modified_dt"]).dt.days
    stale_items = df_final[df_final["days_since_modified"] > stale_cutoff_days].sort_values(
        "days_since_modified", ascending=False
    )
    print(f"Items not modified in over {stale_cutoff_days} days ({len(stale_items)} found):")
    display(stale_items[["name", "type", "created_by", "last_modified", "days_since_modified"]])
else:
    print("Staleness check skipped - Admin API data (last_modified) not available.")

## 7. Optional - persist snapshot to a Delta table for trend tracking

In [ ]:
if save_to_lakehouse:
    try:
        spark_df = spark.createDataFrame(df_final.astype(str))
        spark_df.write.mode("append").format("delta").saveAsTable(lakehouse_table_name)
        print(f"Snapshot appended to Delta table: {lakehouse_table_name}")
    except Exception as e:
        print("Could not save to lakehouse - make sure a default Lakehouse is attached to this notebook.")
        print(f"Details: {e}")
else:
    print("save_to_lakehouse=False - skipping persistence. Set to True (and attach a Lakehouse) to track history over time.")

## Permissions recap

- **Layer 1 (inventory only):** Viewer role on the workspace. Nothing else needed.
- **Layer 2 (Created By / Last Modified):** the identity running this notebook must be a
  **Fabric Administrator** (Microsoft 365 admin role assignment). Workspace Admin/Contributor
  is *not* sufficient for `/v1/admin/*` endpoints.
- **If run under a Service Principal** (e.g. via a pipeline schedule instead of interactively):
  the tenant setting *"Service principals can access read-only admin APIs"* must be enabled,
  and the SPN must be added to the security group that setting is scoped to.
- **Saving to Delta:** Contributor+ role on the workspace, plus a default Lakehouse attached
  to this notebook with write access.

In [ ]:
df = spark.sql("SELECT * FROM LK_Fabric.workspace_inventory_snapshot LIMIT 1000")
display(df)